# Emotion classification with DistilBERT — VS Code + Miniconda
A guided notebook based on our DLH exercises: load → tokenize → pad → train → evaluate → save → predict.

**Setup:** Open this notebook in VS Code, choose **Select Kernel → Python Environments → dlh-ai**. Your environment uses Python 3.11 and CPU PyTorch. Install the VS Code Python and Jupyter extensions. If the kernel is unavailable, run `conda activate dlh-ai` then `python -m pip install ipykernel` in your terminal.

Run cells in order. Training is optional and can take a long time on CPU. This notebook contains no credentials and does not upload anything by default. All Python imports use `import`, not `from ... import ...`.

## Environment and installation
Run installation only if needed, then restart the kernel. `%pip` installs into the selected notebook kernel. These commands are for your local CPU environment; do not use the CPU PyTorch command on Colab GPU.

In [1]:
# Uncomment only if the packages are missing.
# %pip install --no-cache-dir torch==2.8.0+cpu --index-url https://download.pytorch.org/whl/cpu
# %pip install transformers==4.56.1 datasets==4.0.0 accelerate==1.12.0 sentence-transformers==5.1.0 tiktoken==0.11.0 sentencepiece==0.2.1 tabulate==0.9.0 scikit-learn pandas matplotlib ipykernel

In [2]:
import sys
import pathlib
import importlib.metadata
import torch
import transformers
import datasets
import numpy as np
import pandas as pd
import sklearn.metrics
import matplotlib.pyplot as plt
import huggingface_hub

print("Interpreter:", sys.executable)
for package in ["torch", "transformers", "datasets", "accelerate", "scikit-learn"]:
    print(package, importlib.metadata.version(package))
print("CUDA build:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("Working directory:", pathlib.Path.cwd())

c:\Users\ranja\miniconda3\envs\dlh-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Interpreter: c:\Users\ranja\miniconda3\envs\dlh-ai\python.exe
torch 2.8.0+cpu
transformers 4.56.1
datasets 4.0.0
accelerate 1.12.0
scikit-learn 1.6.1
CUDA build: None
GPU available: False
Working directory: c:\Users\ranja\OneDrive\Desktop\AI_Academy\DLH\dlh-modern_ai\deep_learning\finetuning_llm


## 0. Load and inspect the dataset
The configuration named `split` contains predefined train, validation, and test partitions. A configuration name is different from the argument `split="train"`. Leaving that argument out loads all partitions. The task has six emotion classes, rather than positive/negative sentiment alone.

In [3]:
def load_emotion_dataset():
    return datasets.load_dataset("dair-ai/emotion", "split")

print("Configurations:", datasets.get_dataset_config_names("dair-ai/emotion"))
dataset = load_emotion_dataset()
print(dataset)
print(dataset["train"][0])
label_names = dataset["train"].features["label"].names
id2label = dict(enumerate(label_names))
label2id = {name: idx for idx, name in id2label.items()}
print("Labels:", id2label)
display(pd.DataFrame({"split": list(dataset), "rows": [len(dataset[s]) for s in dataset]}))

Configurations: ['split', 'unsplit']
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}
Labels: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


,split,rows
0,train,16000
1,validation,2000
2,test,2000


## 1. Load pretrained DistilBERT
The tokenizer converts text to token IDs. The model has one output per emotion. Newly initialized classification weights are expected: the base model has not learned this emotion task yet. Seed before loading so the new head is reproducible.

In [4]:
def load_distilbert(model_name, num_classes, id2label, label2id):
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    model = transformers.AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes, id2label=id2label, label2id=label2id
    )
    return tokenizer, model

transformers.set_seed(0)
tokenizer, model = load_distilbert(
    "distilbert-base-uncased", len(label_names), id2label, label2id
)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum length:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Vocabulary size: 30522
Maximum length: 512
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## 2. Tokenize all splits
Truncation caps token length; batched mapping processes multiple texts at once. Return three Dataset objects to match the exercise main scripts. Unpacking a DatasetDict directly would instead produce its string keys.

In [5]:
def tokenize_and_map(dataset, tokenizer, max_length, truncation, batched):
    def tokenize_examples(examples):
        return tokenizer(examples["text"], max_length=max_length, truncation=truncation)
    tokenized = dataset.map(tokenize_examples, batched=batched)
    return tokenized["train"], tokenized["validation"], tokenized["test"]

tokenized_train, tokenized_val, tokenized_test = tokenize_and_map(
    dataset, tokenizer, max_length=128, truncation=True, batched=True
)
print(tokenized_train[0])

Map: 100%|██████████| 2000/2000 [00:00<00:00, 22724.73 examples/s]

{'text': 'i didnt feel humiliated', 'label': 0, 'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


## 3. Dynamic padding
Each batch is padded only to its longest sequence. An attention mask marks real tokens and padding. Trainer removes unused text columns automatically; when demonstrating the collator directly, omit text manually.

In [6]:
def create_data_collator(tokenizer):
    return transformers.DataCollatorWithPadding(
        tokenizer=tokenizer, padding="longest", return_tensors="pt"
    )

data_collator = create_data_collator(tokenizer)
examples = [{k: v for k, v in tokenized_train[i].items() if k != "text"}
            for i in range(3)]
batch = data_collator(examples)
print({key: tuple(value.shape) for key, value in batch.items()})

{'input_ids': (3, 23), 'attention_mask': (3, 23), 'labels': (3,)}


## 4. Evaluation metrics
Argmax selects the highest logit; softmax is unnecessary for selecting a class. Weighted metrics weight classes by their true sample counts and can hide weak performance on rare classes. Inspect the per-class report too. Functions belong to `sklearn.metrics`, not the top-level `sklearn` module.

In [7]:
def compute_metrics(predictions):
    logits = predictions.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = sklearn.metrics.precision_recall_fscore_support(
        predictions.label_ids, predicted, average="weighted", zero_division=0
    )
    return {"accuracy": float(sklearn.metrics.accuracy_score(predictions.label_ids, predicted)),
            "precision": float(precision), "recall": float(recall), "f1": float(f1)}

## 5. Configure training
Transformers 4.56.1 uses `eval_strategy`. Matching evaluation and saving strategies allow loading the best checkpoint at the end. Higher F1 is better. The exercise function enables Hub uploads; the local workflow explicitly disables uploads afterward. `report_to="none"` avoids external experiment-tracker login prompts.

In [17]:
def configure_training_args(output_dir, epochs, per_device_train_batch_size,
                            per_device_eval_batch_size, learning_rate, weight_decay,
                            metric_for_best_model, seed):
    return transformers.TrainingArguments(
        output_dir=output_dir, num_train_epochs=epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        learning_rate=learning_rate, weight_decay=weight_decay,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model=metric_for_best_model,
        greater_is_better=True, push_to_hub=True, seed=seed,
        report_to="none", save_total_limit=2
    )

# Start with a small smoke run. Change to False for the full five-epoch exercise.
SMOKE_RUN = True
RUN_TRAINING = False  # Set True when ready to start.
SAVE_DIR = pathlib.Path.cwd() / "hbtn_emotion_classifier_best_model"
training_args = configure_training_args(
    "./hbtn_emotion_classifier", 1 if SMOKE_RUN else 5,
    16, 32, 2e-5, 0.01, "f1", 0
)
training_args.push_to_hub = False
training_args.use_cpu = True
train_input = tokenized_train.select(range(256)) if SMOKE_RUN else tokenized_train
val_input = tokenized_val.select(range(128)) if SMOKE_RUN else tokenized_val
print("Training examples:", len(train_input), "Epochs:", training_args.num_train_epochs)

Training examples: 16000 Epochs: 5


## 6. Train, evaluate, and save
Validation selects the best checkpoint. Test data is used only after training. The default evaluation prefix is `eval_`, matching the exercise main script. Smoke-run results only check that the workflow works; they are not representative final performance.

In [20]:
def train_model(model, training_args, train_dataset, eval_dataset, test_dataset,
                tokenizer, data_collator, compute_metrics, model_save_name):
    trainer = transformers.Trainer(
        model=model, args=training_args, train_dataset=train_dataset,
        eval_dataset=eval_dataset, processing_class=tokenizer,
        data_collator=data_collator, compute_metrics=compute_metrics
    )
    train_results = trainer.train()
    test_results = trainer.evaluate(eval_dataset=test_dataset)
    trainer.save_model(model_save_name)
    tokenizer.save_pretrained(model_save_name)
    return trainer, train_results, test_results

trainer = None
if RUN_TRAINING:
    trainer, train_results, test_results = train_model(
        model, training_args, train_input, val_input, tokenized_test,
        tokenizer, data_collator, compute_metrics, str(SAVE_DIR)
    )
    print("Training metrics:", train_results.metrics)
    display(pd.DataFrame([test_results]))
    print("Saved model:", SAVE_DIR)
else:
    print("Training skipped. Set RUN_TRAINING=True above, rerun that cell, then this cell.")

c:\Users\ranja\miniconda3\envs\dlh-ai\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Detailed test evaluation
Rows are actual labels; columns are predicted labels. The diagonal counts correct predictions. Support is the number of true examples per class. Explicit label IDs keep the matrix and report aligned even if a class is absent in a subset.

In [10]:
if trainer is not None:
    pred_output = trainer.predict(tokenized_test)
    logits = pred_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    pred_labels = np.argmax(logits, axis=-1)
    true_labels = pred_output.label_ids
    label_ids = list(range(len(label_names)))
    matrix = sklearn.metrics.confusion_matrix(true_labels, pred_labels, labels=label_ids)
    display(pd.DataFrame(matrix, index=label_names, columns=label_names))
    print(sklearn.metrics.classification_report(
        true_labels, pred_labels, labels=label_ids,
        target_names=label_names, digits=4, zero_division=0
    ))
    fig, ax = plt.subplots(figsize=(8, 7))
    sklearn.metrics.ConfusionMatrixDisplay(matrix, display_labels=label_names).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45
    )
    ax.set_title("Emotion classification — test set")
    plt.tight_layout()
    plt.show()
else:
    print("Train first, or load a saved model with the next cell.")

Train first, or load a saved model with the next cell.


## 7. Inference from local files or the Hub
Use the local saved folder, or set MODEL_SOURCE to your exact Hub ID, e.g. `yourname/hbtn_emotion_classifier`. A nonexistent local path can produce a misleading repository validation error. Training in Colab does not create model files on your Windows computer. For a private Hub repository, authenticate first.

In [18]:
def inference_mode(model_path, top_k):
    return transformers.pipeline(
        "text-classification", model=model_path, tokenizer=model_path,
        top_k=top_k, truncation=True, device=-1
    )

MODEL_SOURCE = "ranjanasingh/hbtn_emotion_classifier"
if MODEL_SOURCE == str(SAVE_DIR) and not (SAVE_DIR / "config.json").is_file():
    print("Local model not found:", SAVE_DIR)
    print("Train first, download the complete model folder, or enter your Hub model ID.")
else:
    emotion_classifier = inference_mode(MODEL_SOURCE, top_k=3)
    texts = ["I feel really so disappointed.",
             "I absolutely love spending time with my family.",
             "This is so frustrating! I'm really angry!",
             "I love this new song!", "I'm petrified about what might happen."]
    predictions = emotion_classifier(texts)
    display(pd.DataFrame([
        {"text": text, "rank": rank, "emotion": p["label"], "score": p["score"]}
        for text, result in zip(texts, predictions)
        for rank, p in enumerate(result, start=1)
    ]))

c:\Users\ranja\miniconda3\envs\dlh-ai\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ranja\.cache\huggingface\hub\models--ranjanasingh--hbtn_emotion_classifier. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back 

,text,rank,emotion,score
0,I feel really so disappointed.,1,sadness,0.994225
1,I feel really so disappointed.,2,anger,0.002141
2,I feel really so disappointed.,3,joy,0.001468
3,I absolutely love spending time with my family.,1,love,0.600378
4,I absolutely love spending time with my family.,2,joy,0.371137
5,I absolutely love spending time with my family.,3,anger,0.013637
6,This is so frustrating! I'm really angry!,1,anger,0.990620
7,This is so frustrating! I'm really angry!,2,sadness,0.003120
8,This is so frustrating! I'm really angry!,3,fear,0.002855
9,I love this new song!,1,joy,0.872963


## Optional: evaluate a saved model without retraining
This is useful after restarting the kernel. Run setup, dataset loading, tokenization, and collator cells first. Set MODEL_SOURCE in the inference cell to the trained model location.

In [19]:
# Uncomment to evaluate a saved local/Hub model.
saved_model = transformers.AutoModelForSequenceClassification.from_pretrained(MODEL_SOURCE)
saved_tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_SOURCE)
trainer = transformers.Trainer(
     model=saved_model,
     args=transformers.TrainingArguments(output_dir="./evaluation", report_to="none", use_cpu=True),
     processing_class=saved_tokenizer,
     data_collator=create_data_collator(saved_tokenizer),
     compute_metrics=compute_metrics
 )
print(trainer.evaluate(eval_dataset=tokenized_test))
# Then rerun the detailed test evaluation cell.
# This assumes the saved model uses the same DistilBERT tokenizer and label mappings.

{'eval_loss': 0.20308980345726013, 'eval_model_preparation_time': 0.0019, 'eval_accuracy': 0.9235, 'eval_precision': 0.926511010103497, 'eval_recall': 0.9235, 'eval_f1': 0.9231016736606877, 'eval_runtime': 27.0239, 'eval_samples_per_second': 74.009, 'eval_steps_per_second': 9.251}


## Optional: upload your final model
Use a write token. Never paste a token into notebook source or output. Login prompts keep the token out of this notebook. Upload is explicitly disabled until you opt in. Select a repository you intend to create or update. Public repositories are visible to everyone; use private=True for a new private repository.

In [26]:
UPLOAD_TO_HUB = False
MODEL_SOURCE = "ranjanasingh/hbtn_emotion_classifier"

In [27]:
import transformers

repo_id = "ranjanasingh/hbtn_emotion_classifier"

saved_model = transformers.AutoModelForSequenceClassification.from_pretrained(
    repo_id
)
saved_tokenizer = transformers.AutoTokenizer.from_pretrained(repo_id)

SAVE_DIR.mkdir(parents=True, exist_ok=True)
saved_model.save_pretrained(str(SAVE_DIR))
saved_tokenizer.save_pretrained(str(SAVE_DIR))

print("Config exists:", (SAVE_DIR / "config.json").is_file())

Config exists: True


In [25]:
print("Model folder:", SAVE_DIR)
print("Config exists:", (SAVE_DIR / "config.json").is_file())

Model folder: c:\Users\ranja\OneDrive\Desktop\AI_Academy\DLH\dlh-modern_ai\deep_learning\finetuning_llm\hbtn_emotion_classifier_best_model
Config exists: False


In [ ]:
import getpass

UPLOAD_TO_HUB = True
REPO_ID = "ranjanasingh/hbtn_emotion_classifier"
if UPLOAD_TO_HUB:
    if "YOUR_USERNAME" in REPO_ID:
        raise ValueError("Set REPO_ID to your own Hugging Face repository ID.")
    if not (SAVE_DIR / "config.json").is_file():
        raise FileNotFoundError("Save the trained model locally before uploading.")
    hf_token = getpass.getpass("Hugging Face write token: ")
    huggingface_hub.login(token=hf_token, add_to_git_credential=False)
    del hf_token
    upload_model = transformers.AutoModelForSequenceClassification.from_pretrained(str(SAVE_DIR))
    upload_tokenizer = transformers.AutoTokenizer.from_pretrained(str(SAVE_DIR))
    upload_model.push_to_hub(REPO_ID, private=True)
    upload_tokenizer.push_to_hub(REPO_ID, private=True)
    print("Model repository:", "https://huggingface.co/" + REPO_ID)
else:
    print("Upload skipped. Set your repository ID and UPLOAD_TO_HUB=True to upload.")

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.